# 🌿 Training Pipeline: ConvNeXt-Tiny (2-Phase Transfer Learning + ONNX Export)

- **Mục tiêu:** Đạt Macro F1-Score & Accuracy tối ưu trên 38 lớp PlantVillage, đồng thời xuất bản **ONNX** để tối ưu độ trễ (latency) khi cắm vào FastAPI (`backend/service.py`).
- **Môi trường khuyên dùng:** Google Colab GPU **T4** (tự động mount Google Drive, giải nén data lên SSD tạm `/content/data` để train nhanh, và lưu checkpoint `.pth` + `.onnx` thẳng về Drive để máy local tự động nhận).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import sys
from pathlib import Path
import torch

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Mac_PhamGiaKhoi/Dự án/ML/plant-disease-ml")
    assert PROJECT_DIR.exists(), f"Không tìm thấy thư mục dự án trên Drive: {PROJECT_DIR}"
    os.chdir(PROJECT_DIR)
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"IN_COLAB   : {IN_COLAB}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DEVICE     : {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB   : True
PROJECT_DIR: /content/drive/MyDrive/Mac_PhamGiaKhoi/Dự án/ML/plant-disease-ml
DEVICE     : cuda (Tesla T4)


In [4]:
# Chuẩn bị dữ liệu nhanh:
# Trên Colab, đọc 87k ảnh trực tiếp từ Google Drive rất chậm -> giải nén vào ổ SSD tạm /content/data.
if IN_COLAB:
    DATA_DIR = Path("/content/data")
    if not (DATA_DIR / "train").exists():
        from scripts.download_data import DEFAULT_GDRIVE_FILE_ID, download_from_gdrive, extract_and_organize

        drive_archive = PROJECT_DIR / "data.tar.gz"
        if drive_archive.exists():
            print(f"📦 Giải nén trực tiếp từ {drive_archive} sang {DATA_DIR}...")
            extract_and_organize(drive_archive, DATA_DIR)
        else:
            tmp_archive = Path("/content/data.tar.gz")
            print("🌐 Tải data.tar.gz vào SSD tạm của Colab...")
            download_from_gdrive(DEFAULT_GDRIVE_FILE_ID, tmp_archive)
            extract_and_organize(tmp_archive, DATA_DIR)
            tmp_archive.unlink(missing_ok=True)
    else:
        print(f"✅ Đã có sẵn dữ liệu trên SSD Colab: {DATA_DIR}")
else:
    DATA_DIR = PROJECT_DIR / "data"
    assert (DATA_DIR / "train").exists(), "Chưa có thư mục data/train. Hãy chạy python scripts/download_data.py trước."

print("DATA_DIR:", DATA_DIR)

🌐 Tải data.tar.gz vào SSD tạm của Colab...
🌐 Đang kết nối Google Drive để tải file về /content/data.tar.gz...


Downloading...
From (original): https://drive.google.com/uc?id=1HKKtjCbc0BcdcNAMA4ualjxXhOprBaqc
From (redirected): https://drive.google.com/uc?id=1HKKtjCbc0BcdcNAMA4ualjxXhOprBaqc&confirm=t&uuid=4143e2f4-3f21-4799-a71d-909179168d40
To: /content/data.tar.gz
100%|██████████| 1.38G/1.38G [00:15<00:00, 88.0MB/s]


📦 Đang giải nén file tar.gz: data.tar.gz...
🚚 Đang quét và phân phối các tập dữ liệu vào đúng vị trí data/...
  ✅ Đã sắp xếp: data/train/
  ✅ Đã sắp xếp: data/valid/
  ✅ Đã sắp xếp: data/test/
🎉 Hoàn tất! Toàn bộ dữ liệu đã nằm chuẩn chỉnh trong thư mục data/
DATA_DIR: /content/data


In [5]:
# Cài onnx / onnxruntime trên Colab nếu chưa có
if IN_COLAB:
    %pip install -q onnx onnxruntime

# ==================== CẤU HÌNH THỬ NGHIỆM ====================
# Hỗ trợ: 'convnext_tiny' (khuyên dùng), 'efficientnet_b0', 'efficientnet_v2_s',
#         'resnet50', 'mobilenet_v3_large', 'swin_t'
MODEL_NAME = "convnext_tiny"
AUTHOR = "khoi"

BATCH_SIZE = 64
IMAGE_SIZE = 224
PHASE1_EPOCHS = 2      # Đóng băng backbone, chỉ train head
PHASE2_EPOCHS = 12     # Mở băng toàn bộ với differential LR
PHASE1_LR = 1e-3
PHASE2_BACKBONE_LR = 1e-4
PHASE2_HEAD_LR = 5e-4
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = 0.1
PATIENCE = 4           # Early stopping theo val_macro_f1
SEED = 42

SAVE_PTH = PROJECT_DIR / "models" / f"{MODEL_NAME}_{AUTHOR}.pth"
SAVE_ONNX = PROJECT_DIR / "models" / f"{MODEL_NAME}_{AUTHOR}.onnx"

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Checkpoint đích (tự sync về Mac): {SAVE_PTH}")
print(f"ONNX đích (tự sync về Mac)      : {SAVE_ONNX}")

Checkpoint đích (tự sync về Mac): /content/drive/MyDrive/Mac_PhamGiaKhoi/Dự án/ML/plant-disease-ml/models/convnext_tiny_khoi.pth
ONNX đích (tự sync về Mac)      : /content/drive/MyDrive/Mac_PhamGiaKhoi/Dự án/ML/plant-disease-ml/models/convnext_tiny_khoi.onnx


In [6]:
# 1. Dùng chung Data Pipeline chuẩn của nhóm
import json
from notebooks.data_pipeline import build_dataloaders
from notebooks.eval_engine import evaluate_model, print_evaluation_summary

train_loader, valid_loader, idx_to_class, class_weights = build_dataloaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)

# Kiểm tra khớp 100% với models/class_indices.json
with open(PROJECT_DIR / "models" / "class_indices.json", encoding="utf-8") as f:
    ref_map = {int(k): v for k, v in json.load(f)["idx_to_class"].items()}
assert idx_to_class == ref_map, "Thứ tự lớp không khớp với models/class_indices.json!"

num_classes = len(idx_to_class)
print(f"✅ Đã nạp {num_classes} lớp | Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)}")

✅ Đã nạp 38 lớp | Train batches: 1098 | Valid batches: 275


In [7]:
# 2. Khởi tạo Pretrained Backbone & tách Head để fine-tune 2 giai đoạn
import torch.nn as nn
from torchvision import models

def build_pretrained_model(model_name: str, num_classes: int):
    """Trả về (model, head_module) khớp 100% với backend/service.py."""
    if model_name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
        return m, m.classifier
    if model_name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m, m.classifier
    if model_name == "efficientnet_v2_s":
        m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m, m.classifier
    if model_name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m, m.fc
    if model_name == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
        return m, m.classifier
    if model_name == "swin_t":
        m = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)
        m.head = nn.Linear(m.head.in_features, num_classes)
        return m, m.head
    raise ValueError(f"Không hỗ trợ model_name='{model_name}'")

model, head_module = build_pretrained_model(MODEL_NAME, num_classes)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Khởi tạo {MODEL_NAME}: {total_params:,} tham số (~{total_params * 4 / 1024**2:.1f} MB FP32)")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 197MB/s]


✅ Khởi tạo convnext_tiny: 27,849,350 tham số (~106.2 MB FP32)


In [8]:
# 3. Huấn luyện 2 giai đoạn với Mixed Precision (AMP) + Label Smoothing + Early Stopping
import copy
import time
import torch.optim as optim

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=LABEL_SMOOTHING,
)
use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)
    return running_loss / total, correct / total

best_macro_f1 = 0.0
best_metrics = None
no_improve = 0
SAVE_PTH.parent.mkdir(parents=True, exist_ok=True)

def save_if_best(metrics):
    global best_macro_f1, best_metrics, no_improve
    if metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = metrics["macro_f1"]
        best_metrics = copy.deepcopy(metrics)
        no_improve = 0
        torch.save({
            "model_name": MODEL_NAME,
            "author": AUTHOR,
            "num_classes": num_classes,
            "val_accuracy": metrics["accuracy"],
            "val_f1_macro": metrics["macro_f1"],
            "metrics": metrics,
            "model_state_dict": model.state_dict(),
        }, SAVE_PTH)
        print(f"   ⭐ Lưu Best Checkpoint -> {SAVE_PTH.name} (Macro-F1: {best_macro_f1:.4f})")
    else:
        no_improve += 1

# --- PHASE 1: Warmup Head (đóng băng backbone) ---
print(f"\n=== PHASE 1: WARMUP HEAD ({PHASE1_EPOCHS} epochs) ===")
for p in model.parameters():
    p.requires_grad = False
for p in head_module.parameters():
    p.requires_grad = True

opt_p1 = optim.AdamW(head_module.parameters(), lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
for ep in range(1, PHASE1_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, opt_p1)
    val_m = evaluate_model(model, valid_loader, device=device, criterion=criterion)
    print(
        f"[P1 {ep}/{PHASE1_EPOCHS}] ({time.time()-t0:.1f}s) | "
        f"Train Loss {tr_loss:.4f} Acc {tr_acc*100:.2f}% | "
        f"Val Loss {val_m['val_loss']:.4f} Acc {val_m['accuracy']*100:.2f}% Macro-F1 {val_m['macro_f1']*100:.2f}%"
    )
    save_if_best(val_m)

# --- PHASE 2: Full Fine-tuning với Differential LR + CosineAnnealingLR ---
print(f"\n=== PHASE 2: FULL FINE-TUNING ({PHASE2_EPOCHS} epochs) ===")
for p in model.parameters():
    p.requires_grad = True

head_ids = {id(p) for p in head_module.parameters()}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
opt_p2 = optim.AdamW([
    {"params": backbone_params, "lr": PHASE2_BACKBONE_LR},
    {"params": head_module.parameters(), "lr": PHASE2_HEAD_LR},
], weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt_p2, T_max=PHASE2_EPOCHS, eta_min=1e-6)
no_improve = 0

for ep in range(1, PHASE2_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, opt_p2)
    scheduler.step()
    val_m = evaluate_model(model, valid_loader, device=device, criterion=criterion)
    print(
        f"[P2 {ep:02d}/{PHASE2_EPOCHS:02d}] ({time.time()-t0:.1f}s) | "
        f"Train Loss {tr_loss:.4f} Acc {tr_acc*100:.2f}% | "
        f"Val Loss {val_m['val_loss']:.4f} Acc {val_m['accuracy']*100:.2f}% Macro-F1 {val_m['macro_f1']*100:.2f}%"
    )
    save_if_best(val_m)
    if no_improve >= PATIENCE:
        print(f"⏹️ Early stopping sau {PATIENCE} epochs không cải thiện Macro-F1.")
        break


=== PHASE 1: WARMUP HEAD (2 epochs) ===
[P1 1/2] (492.2s) | Train Loss 1.0792 Acc 92.72% | Val Loss 0.8913 Acc 97.51% Macro-F1 97.50%
   ⭐ Lưu Best Checkpoint -> convnext_tiny_khoi.pth (Macro-F1: 0.9750)
[P1 2/2] (486.6s) | Train Loss 0.8828 Acc 97.47% | Val Loss 0.8546 Acc 98.32% Macro-F1 98.30%
   ⭐ Lưu Best Checkpoint -> convnext_tiny_khoi.pth (Macro-F1: 0.9830)

=== PHASE 2: FULL FINE-TUNING (12 epochs) ===
[P2 01/12] (564.2s) | Train Loss 0.7571 Acc 99.20% | Val Loss 0.7124 Acc 99.49% Macro-F1 99.48%
   ⭐ Lưu Best Checkpoint -> convnext_tiny_khoi.pth (Macro-F1: 0.9948)
[P2 02/12] (576.1s) | Train Loss 0.6992 Acc 99.66% | Val Loss 0.6942 Acc 99.60% Macro-F1 99.58%
   ⭐ Lưu Best Checkpoint -> convnext_tiny_khoi.pth (Macro-F1: 0.9958)
[P2 03/12] (583.4s) | Train Loss 0.6933 Acc 99.61% | Val Loss 0.6921 Acc 99.54% Macro-F1 99.53%
[P2 04/12] (556.8s) | Train Loss 0.6863 Acc 99.71% | Val Loss 0.6860 Acc 99.71% Macro-F1 99.71%
   ⭐ Lưu Best Checkpoint -> convnext_tiny_khoi.pth (Macro-F1

KeyboardInterrupt: 

In [9]:
# 4. Đánh giá chính thức Best Checkpoint bằng eval_engine.py
ckpt = torch.load(SAVE_PTH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
final_metrics = evaluate_model(model, valid_loader, device=device, criterion=criterion)
print_evaluation_summary(final_metrics, model_name=f"{MODEL_NAME}_{AUTHOR}")


 📊 BẢNG ĐÁNH GIÁ CHUẨN: CONVNEXT_TINY_KHOI
 🎯 Macro F1-Score (Tiêu chí chính) : 99.95%
 📈 Accuracy                         : 99.95%
 ⚖️  Weighted F1-Score               : 99.95%
 🔍 Macro Precision / Recall         : 99.95% / 99.95%
-------------------------------------------------------
 ⚡ Độ trễ suy luận (Latency)        : 0.26 ms/ảnh
 🚀 Tốc độ xử lý (Throughput)        : 3814.9 khung hình/giây (FPS)
 📦 Dung lượng mô hình               : 106.24 MB (27,849,350 tham số)



In [10]:
# 5. Xuất sang ONNX & Benchmark Latency PyTorch vs ONNX Runtime (CPU)
import numpy as np
import onnxruntime as ort

model_cpu = model.to("cpu").eval()
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)

torch.onnx.export(
    model_cpu,
    dummy,
    str(SAVE_ONNX),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
    dynamo=False,
)
print(f"✅ Đã xuất ONNX: {SAVE_ONNX} ({SAVE_ONNX.stat().st_size / 1024**2:.1f} MB)")

# Kiểm tra độ khớp output (Parity Check)
sess = ort.InferenceSession(str(SAVE_ONNX), providers=["CPUExecutionProvider"])
with torch.no_grad():
    pt_out = model_cpu(dummy).numpy()
ort_out = sess.run(None, {"input": dummy.numpy()})[0]
max_diff = float(np.max(np.abs(pt_out - ort_out)))
print(f"🔍 Max abs diff (PyTorch vs ONNX): {max_diff:.2e}")
assert max_diff < 1e-3, f"Sai lệch ONNX quá lớn: {max_diff}"

# Đo tốc độ 1 ảnh trên CPU (30 lần chạy sau warmup)
for _ in range(5):
    with torch.no_grad():
        model_cpu(dummy)
    sess.run(None, {"input": dummy.numpy()})

t0 = time.perf_counter()
for _ in range(30):
    with torch.no_grad():
        model_cpu(dummy)
pt_ms = (time.perf_counter() - t0) / 30 * 1000

t0 = time.perf_counter()
for _ in range(30):
    sess.run(None, {"input": dummy.numpy()})
ort_ms = (time.perf_counter() - t0) / 30 * 1000

print(f"⚡ CPU Latency 1 ảnh — PyTorch: {pt_ms:.1f} ms | ONNX Runtime: {ort_ms:.1f} ms (nhanh hơn {pt_ms / ort_ms:.2f}x)")

/tmp/ipykernel_902/943556609.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✅ Đã xuất ONNX: /content/drive/MyDrive/Mac_PhamGiaKhoi/Dự án/ML/plant-disease-ml/models/convnext_tiny_khoi.onnx (106.3 MB)
🔍 Max abs diff (PyTorch vs ONNX): 1.07e-06
⚡ CPU Latency 1 ảnh — PyTorch: 173.0 ms | ONNX Runtime: 118.9 ms (nhanh hơn 1.46x)


In [11]:
# 6. Nén ONNX INT8 Dynamic Quantization & Benchmark FP32 vs INT8
import importlib
importlib.invalidate_caches()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

SAVE_ONNX_INT8 = PROJECT_DIR / "models" / f"{MODEL_NAME}_{AUTHOR}_int8.onnx"
from scripts.quantize_onnx import quantize_model, benchmark_session, evaluate_onnx_on_valid

quantize_model(SAVE_ONNX, SAVE_ONNX_INT8)
fp32_mb = SAVE_ONNX.stat().st_size / 1024**2
int8_mb = SAVE_ONNX_INT8.stat().st_size / 1024**2

dummy_np = dummy.numpy()
fp32_out, fp32_ms = benchmark_session(SAVE_ONNX, dummy_np, runs=30)
int8_out, int8_ms = benchmark_session(SAVE_ONNX_INT8, dummy_np, runs=30)
a, b = fp32_out.reshape(-1), int8_out.reshape(-1)
cos_sim = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

print(f"✅ Đã nén INT8: {SAVE_ONNX_INT8.name}")
print(f"📦 Dung lượng : {fp32_mb:.1f} MB -> {int8_mb:.1f} MB (giảm {fp32_mb / int8_mb:.2f}x)")
print(f"⚡ CPU Latency: {fp32_ms:.1f} ms -> {int8_ms:.1f} ms (nhanh hơn {fp32_ms / int8_ms:.2f}x)")
print(f"🎯 Cosine Sim : {cos_sim:.6f}")

# Kiểm chứng nhanh độ chính xác INT8 trên 50 batch (1.600 ảnh valid)
int8_val = evaluate_onnx_on_valid(SAVE_ONNX_INT8, str(DATA_DIR), batch_size=32, max_batches=50)
print(f"📊 INT8 Valid ({int8_val['samples']} ảnh): Acc {int8_val['accuracy']*100:.2f}% | Macro-F1 {int8_val['macro_f1']*100:.2f}%")

✅ Đã nén INT8: convnext_tiny_khoi_int8.onnx
📦 Dung lượng : 106.3 MB -> 32.3 MB (giảm 3.29x)
⚡ CPU Latency: 113.1 ms -> 151.1 ms (nhanh hơn 0.75x)
🎯 Cosine Sim : 0.999192
📊 INT8 Valid (1600 ảnh): Acc 100.00% | Macro-F1 100.00%
